# 76. 코사인 원판 모델을 주 모델 또는 보조 모델로 사용하면?
`A+B`에서는 A가 추적한 좌표가 비었을 때만 B가 채웁니다. 시간 연결·화면 경계 규칙은 두 모델 모두에 적용합니다. 추론과 집계는 프로젝트 루트에서 `python -m wlcv.evaluation.plate_cosine_fallback`으로 실행합니다. 아래 셀은 그 결과를 읽습니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from wlcv.evaluation.plate_cosine_fallback import MODELS, OUTPUT
summary = pd.read_csv(OUTPUT / "summary.csv").set_index("case")
athletes = pd.read_csv(OUTPUT / "athletes.csv")
frames = pd.read_csv(OUTPUT / "frames.csv", dtype={"match": "boolean"})
backgrounds = pd.read_csv(OUTPUT / "backgrounds.csv")
display(summary[["fn","fp","gaps","longest","gained","lost","rises","rise_mae","bg_fp"]].round(2))
display(backgrounds.pivot(index="case",columns="video",values="fp"))

## 우선순위에 따른 차이
파란색은 코사인이 주 모델인 경우, 주황색은 코사인을 보조로 쓴 경우입니다. 왼쪽 그림은 검증 동작의 미탐, 오른쪽 그림은 학습에서 본 빈 배경 두 편에서 신호가 잘못 나온 프레임 수입니다. 점선은 기존 Mac + CutMix 결과입니다.

In [ ]:
partners = [name for name in MODELS if name != "cosine"]
fig, axes = plt.subplots(1, 2, figsize=(10,5), layout="constrained")
for metric, title, ax in zip(["fn","bg_fp"],["Misses in motion (270 labeled)","False positives on training backgrounds (176)"],axes):
    values = pd.DataFrame({
        "Cosine first":[summary.at["cosine+" + p, metric] for p in partners],
        "Cosine second":[summary.at[p + "+cosine", metric] for p in partners]}, index=partners)
    values.plot.barh(ax=ax, color=["#326494","#c47a37"])
    ax.axvline(summary.at["mac+cutmix",metric], color="black",ls="--", label="Mac + CutMix")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Frames")
    ax.legend(fontsize=8)
fig.savefig(OUTPUT / "cosine_pairs.png",dpi=150)
plt.show()

## 동작 중 새로 얻고 잃은 프레임과 선수별 허브 상승량
`selected`에 첫 표의 조합 이름을 입력합니다. 수동 박스에 대한 정탐은 IoU 0.50 이상입니다. 상승량은 무릎 통과부터 접촉까지 모델 박스 중심으로 계산하며, 양의 오차는 수동 상승량보다 크게 측정했다는 뜻입니다.

In [ ]:
selected = "cosine+mac"
baseline = "mac+cutmix"
motion = frames[frames.in_motion & frames.match.notna()]
matches = motion.pivot(index=["video_id","frame_index"],columns="case",values="match")
display(matches.loc[matches[baseline] != matches[selected],[baseline,selected]])
subset = athletes[athletes.case.isin([baseline,selected])]
display(subset[["case","video_id","gap_indices","longest_gap","manual_rise_px","rise_error_px"]].round(2))

## 원본을 확대해 확인하기
초록 점선은 수동 박스, 주황 실선은 선택된 박스입니다. 제목에 실제 좌표를 낸 모델을 표시합니다. `samples`와 `modes`를 바꿔 같은 프레임의 정탐·오탐·누락을 비교할 수 있습니다.

In [ ]:
from matplotlib.patches import Rectangle
from wlcv.config import VIDEOS
from wlcv.data.frames import read_rgb
samples = [("a03_sn_f_01",10),("b02_sn_h_02",35)]
modes = ["mac+cutmix","cosine",selected]
fig, axes = plt.subplots(len(samples),len(modes),figsize=(10,6), layout="constrained",squeeze=False)
for (video, index), row in zip(samples,axes):
    image = read_rgb(VIDEOS / f"{video}.mp4",index)
    data = frames[(frames.video_id == video) & (frames.frame_index == index)].set_index("case")
    for mode, ax in zip(modes,row):
        sample = data.loc[mode]
        ax.imshow(image)
        x,y,r,b = sample[["gt_x1","gt_y1","gt_x2","gt_y2"]]
        if pd.notna(x):
            ax.add_patch(Rectangle((x,y),r-x,b-y,fill=False,edgecolor="lime",ls="--",lw=2))
            ax.set_xlim(max(0,x-(r-x)/2),min(image.shape[1],r+(r-x)/2))
            ax.set_ylim(min(image.shape[0],b+(b-y)/2),max(0,y-(b-y)/2))
        if pd.notna(sample.x1):
            ax.add_patch(Rectangle((sample.x1,sample.y1),sample.x2-sample.x1,sample.y2-sample.y1,fill=False,edgecolor="darkorange",lw=2))
        ax.set_title(f"{mode} | {video[:3]} #{index}\n{sample.selected_model}")
        ax.axis("off")
fig.savefig(OUTPUT / "pair_frames_local.png",dpi=140)
plt.show()

## 수동 시점 두 장에서만 코사인으로 상승량 읽기
기존 영상 궤적은 보존합니다. 두 시점 모두에서 코사인이 독립적으로 잡은 박스와 기존 추적 박스의 IoU가 0.50 이상일 때만 두 코사인 좌표로 상승량을 냅니다. 한 장이라도 통과하지 못하면 두 시점 모두 기존 모델을 씁니다. 실제 수동 허브는 선택이 끝난 뒤 오차를 계산할 때 씁니다.

In [ ]:
checks = pd.read_csv(OUTPUT / "endpoint_checks.csv")
rise = pd.read_csv(OUTPUT / "endpoint_rises.csv")
display(checks[checks.case == "mac"][["video_id","frame_index","inter_model_iou","accepted"]].round(3))
display(rise[rise.case == "mac"][["video_id","source","manual_rise_px","rise_px","error_px"]].round(2))
display(rise.groupby("case").error_px.apply(lambda errors: errors.abs().mean()).round(2))